## Introduction to Large Language Models (LLMs)

Before we dive into prompt engineering, let's establish a foundational understanding of what Large Language Models (LLMs) are and how they work.

### What is an LLM?
Large Language Models are deep neural networks with billions of tunable parameters that have been trained on huge text corpora using "self supervision". Once this foundational training is complete, the LLM parameters are fine-tuned to complete specific tasks such as question answering, summarization, and machine translation.

At its core, an LLM is a system where "text goes in and text comes out". 
* **Tokens:** LLMs don't read words like we do; they convert words or partial words into numerical IDs called **tokens**.
* **Context Window:** The context window (or context length) is the maximum number of tokens an LLM will accept as input at one time.

### Demystifying LLM Text Generation

The Python cell below is a simple simulation, but it illustrates the core mechanic of how Large Language Models actually work: **autoregressive generation**. 

At their simplest level, LLMs operate on a basic principle: text goes in, and text comes out. However, they do not write entire sentences or paragraphs at once. Instead, they break text down into smaller chunks called **tokens** (which can be whole words, syllables, or even single characters).

Here is what our simulation is demonstrating about the LLM "brain":

1. **The Prompt:** This is your initial input. Everything the model generates is strictly based on this starting text. 
2. **The Prediction:** The model analyzes the prompt and calculates the most mathematically probable *single next token*.
3. **The Recursive Loop:** The model takes that single new token, attaches it to the end of your original prompt, and then feeds that *entire new string* back into itself to predict the next token. 

When you run the Python loop below, you are watching this recursive cycle in action. Every time the text updates, you are seeing the model's "context window" growing larger, token by token, until the task is complete.

In [3]:
import time
import sys
from IPython.display import clear_output

prompt = """Instructions: Write a poem that exactly rhymes with orange

Output:
"""

# The "tokens" the LLM generates one by one
tokens = ["As ", "I ", "sat ", "under ", "a ", "tree ", "looking ", "at ", "a ", "door hinge..."]

current_text = prompt

# Print the prompt first
print(current_text + "█")
time.sleep(1.5)

# Simulate the token-by-token generation
for token in tokens:
    current_text += token
    clear_output(wait=True)
    # The █ character simulates the blinking cursor!
    print(current_text + "█") 
    time.sleep(0.4) # Adjust this number to make it type faster or slower

clear_output(wait=True)
print(current_text)
print("\n[Generation Complete]")

Instructions: Write a poem that exactly rhymes with orange

Output:
As I sat under a tree looking at a door hinge...

[Generation Complete]


### The Transformer Architecture
Modern LLMs rely on the **Transformer** architecture, which excels at calculating the contextual meaning of words. The key feature of transformers is their ability to perform fast, parallelized calculations of contextual relationships—known as "attention scores". This allows the model to understand not just the numerical representation of a word, but its relative position and context within a sentence.

#### A Quick Note on the Architecture Diagrams
If the diagrams below look like a daunting maze of math and boxes, don't worry! You do not need to memorize matrix multiplication, SoftMax functions, or the exact wiring of the neural network to use Large Language Models effectively. 

![attention_score](./attention_score.png)<sup>[1]</sup>
*The key feature of transformers is fast parallelized calculations of contextual relationships (attention scores). The colorful grid matrices are simply the mathematical representation of this attention. The model scores every word against every other word in your prompt to build a numerical map of the context.*

![attention_head](./attention_heads.png)<sup>[2]</sup>
*Attention scores for different aspects of sentence structure can be calculate in parallel using multiple attention “heads”.*

![transformer_arch](./transformer_arch.png)<sup>[2]</sup>
*Dissecting the classic transformer architecture. The complex diagram from the original "Attention Is All You Need" paper  highlights the "Multi-Head Attention" blocks. High-level, this just means the model has multiple "heads" looking at different aspects of sentence structure simultaneously (in parallel). This parallel processing is what makes the Transformer architecture so incredibly fast and powerful compared to older AI models.**The main takeaway:** Transformers are exceptionally good at calculating the contextual meaning of words by paying "attention" to how every word in your prompt relates to every other word.*

[1] “Transformers Explained Visually” Ketan Doshi Towards Data Science (2020)

[2] Vaswani, A., et al. (2017). "Attention is all you need." *arXiv preprint*. [Link to paper](https://arxiv.org/abs/1706.03762)

## Prompt Engineering

A prompt is a set of instructions (including content like text, images, etc) provided to an LLM to perform a task.

Prompt Engineering is the process of crafting such the set of instructions that gets an LLM model to generate the desired outcome (i.e., solving the task).

Components of Well-Structured Prompts
- Role: The role the LLM should adopt.
- Task Description: The specific instruction or question.
- Context: Additional information needed for the task.
- Output Format: How the response should be structured.
- Examples (a.k.a. Few-Shot "Learning") (optional): Sample input/output pairs.

In [1]:
import os
%env HF_HOME={os.environ['SCRATCH']}/HFHOME
%env CUDA_HOME=/home1/apps/nvidia/Linux_aarch64/25.5/cuda/12.9/

env: HF_HOME=/scratch/07980/sli4/HFHOME
env: CUDA_HOME=/home1/apps/nvidia/Linux_aarch64/25.5/cuda/12.9/


### Prepare the model

To prepare the model for the tutorial, let's  download the weights from the [Hugging Face Hub](https://huggingface.co/). We will use the `AutoModelForCausalLM` class from 🤗 Transformers to load the model:

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Mxfp4Config

quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)
model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs)
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [7]:
def generate_response(
    model, 
    tokenizer, 
    user_prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.5,
    max_new_tokens: int = 512
) -> str:
    """Sends a request to a Hugging Face model and returns a response."""
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    do_sample = True

    output_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
   
    input_length = input_ids.shape[1]
    new_tokens = output_ids[:, input_length:]
    
    raw_response = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    if "assistantfinal" in raw_response:
        raw_response = raw_response.split("assistantfinal")[-1]
    elif "</think>" in raw_response:
        raw_response = raw_response.split("</think>")[-1]
        
    final_response = raw_response.replace(tokenizer.eos_token, "")
    return final_response.strip()

### 1. Vague vs. Precise Prompt

Let's see how an LLM's output changes when we start from a vague prompt, then adding a persona/role, and finally specifying the task.

In [8]:
# Vague prompt = vague result
vague_prompt = "Tell me about Albert Einstein."

vague_response = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=vague_prompt, 
)
print(vague_response)

In [11]:
# Add persona (role) to the vague prompt
vague_prompt_with_persona = f"""
You are a high-school physics teacher.
Tell me about Albert Einstein.
"""

response_with_persona = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=vague_prompt_with_persona,
)
print(response_with_persona)

In [13]:
precise_prompt = f"""
You are a high-school physics teacher.
Explain the role Albert Einstein played in the development of physics in 20th century.
Write two paragraphs. 
"""

precise_response = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=precise_prompt
)
print(precise_response)

In the case of the vague prompt, we can see that the output text covers various aspects of Albert Einstein's live without providing depth into any of those. Advancing forward, when we provide a role ("a high-school physics teacher"), the output changes its format and is "tailored" to the role/persona specified. Finally, with the precise prompt, we guide an LLM to produce a specific but more detailed response. While for some cases (e.g., famous personas) we might want to learn both perspectives (in breadth and depth), for AI applications we typically want to narrow down the scope of the task to get more specific outputs.

So, **Best Practice #1: Be specific and direct**.

### 2. Add Context

Any LLM model "knows" only what it was trained on. However, you can incorporate new information by providing it as a context.

In [18]:
# Source: https://tacc.utexas.edu/about/staff-directory/niall-gaffney/
context = """
NIALL GAFFNEY

Data and AI Directorate

Phone: 512-475-9504 | Email: ngaffney@tacc.utexas.edu

Education: B.A., M.A., Ph.D., Astronomy University of Texas at Austin

Niall Gaffney's background primarily revolves around the management and utilization of 
large inhomogeneous scientific datasets. Niall, who earned his B.A., M.A., and Ph.D. 
degrees in astronomy from The University of Texas at Austin, joined TACC in May 2013. 
Most of his focus has been on creating environments to foster better data practices 
from improving metadata, data processing, analysis, and reuse. He focuses on improving 
researchers' data practices to accelerate outcomes and better feed the Machine Learning 
and Artificial Intelligence applications which are becoming more broadly adopted in 
science and engineering research fields. Much of this stems from his 13 years as designer 
and developer for the archives at the Space Telescope Science Institute (STScI), which 
holds the data from the Hubble Space Telescope, Kepler, and James Webb Space Telescope 
missions.

He was also a leader in developing the Hubble Legacy Archive. This project harvested 
the 20+ years of Hubble Space Telescope data to create some of the most sensitive 
astronomical data products available for open research. Before his work at STScI, 
Niall was worked as "the friend of the telescope" for the Hobby Eberly Telescope (HET) 
project at the McDonald Observatory in west Texas. This was the start of his work in 
planning experiments and then cataloging the data the HET produced.
"""

In [21]:
# First let's see what a model "knows" about Niall Gaffney.
# In other words, do NOT provide any context first. 
no_context_prompt = "Who is Niall Gaffney?"

response_wo_context = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=no_context_prompt,
)

print(response_wo_context)

analysisWe need to answer: Who is Niall Gaffney? We should provide a concise answer. We need to check if there is a known person by that name. Possibly a sports figure? Let's search memory: Niall Gaffney is an Irish rugby union player? I recall Niall Gaffney is an Irish rugby player born 1985? He played for Connacht and was a fullback. He also played for Ireland national team? He had a short international career. He played for the Irish national team in 2009? He also played for the Ireland U21. He played for the Irish under-20. He was part of the 2011 Rugby World Cup squad? Wait, Niall Gaffney was a rugby league? Let's recall: Niall Gaffney is an Irish rugby union player, fullback, played for Connacht. He also played for the Ireland national team. He was born in 1985. He played for the Ireland national team 2009. He played for the Irish national team 2010? He also played for the Ireland national team in 2010? He was part of the 2011 Rugby World Cup squad. He also played for the Ireland

In [22]:
# Now, let's add the context.
prompt_with_context = no_context_prompt + f"\nContext: {context}"

response_with_context = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=prompt_with_context,
    temperature=1.0
)

print(response_with_context)

**Niall Gaffney** is a senior data‑science professional working in the Data and AI Directorate at the Texas Advanced Computing Center (TACC). He holds a B.A., M.A., and Ph.D. in astronomy from the University of Texas at Austin and has spent over a decade designing and managing large scientific datasets.  

- **TACC (since May 2013)** – focuses on improving data practices, metadata, processing, analysis, and reuse to accelerate research and support machine‑learning applications.  
- **Space Telescope Science Institute (STScI)** – 13 years designing and developing the archives for Hubble, Kepler, and James Webb, and leading the creation of the Hubble Legacy Archive, which turned 20+ years of telescope data into some of the most sensitive open‑access astronomical products.  
- **Prior roles** – “friend of the telescope” for the Hobby‑Eberly Telescope project at the McDonald Observatory, where he began cataloging and planning experiments for HET data.

Gaffney’s work centers on creating en

We've seen that by providing context, we can "add" new information/knowledge to a model without retraining it. What's more, by providing context we can also reduce the occurrence of hallucinations [1]. However, note that adding the context does NOT guarantee that a model will strictly follow it [1].

Also, adding additional instructions like "generate response based on the context provided" to the prompt can also be helpful.

**Best Practice #2: Add specific context to your prompts when applicable.**

[1] Huyen, C. (2024). Prompt Engineering. In AI Engineering: Building Applications with Foundation Models. (pp. 211-252) O'Reilly Media, Inc.

### 3. Prompt Chaining

Break complex tasks into simpler subtasks. Use each LLM's output as a **context** for the next prompt/step.

In [24]:
topic = "the impact of Albert Einstein on philosophy of 20th century"

# Step 1: Create an outline for the blog post
prompt_step_1 = f"Come up with a 3 point outline for a post about: {topic}"

response_step_1 = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=prompt_step_1,
)

print(response_step_1)

**Outline – “The Impact of Albert Einstein on the Philosophy of the 20th Century”**

1. **Re‑shaping the Ontology of Space–Time**
   - *From Newtonian absolutes to relativistic relativity*  
     - Einstein’s 1905 and 1915 papers dissolve the idea of absolute space and time.  
     - Philosophical consequence: the shift from a static, deterministic universe to a dynamic, relativistic one.
   - *Implications for realism and determinism*  
     - The block‑universe picture raises questions about free will, causality, and temporal becoming.  
     - Debates over whether the relativistic metric is a true feature of reality or merely a useful construct.

2. **Einstein’s Influence on the Philosophy of Science**
   - *Theory‑laden observation and the underdetermination of theory*  
     - Einstein’s insistence that theory shapes observation challenges the notion of “pure” data.  
     - Leads to discussions on the role of metaphysical assumptions in scientific explanation.
   - *Realism vs. i

In [26]:
# Step 2: Write introduction using the outline
prompt_step_2=f"""
Using the following OUTLINE, write an introduction paragraph with 80-100 words.
OUTLINE: {response_step_1}
Hook the reader with a surprising fact in the first sentence.
"""

response_step_2 = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=prompt_step_2,
)

print(response_step_2)

Did you know that the very fabric of our universe, once thought immutable, was shattered by a single equation? Einstein’s 1905 and 1915 papers not only replaced Newton’s absolutes with a relativistic spacetime but also forced philosophers to rethink realism, determinism, and the nature of observation. By insisting that theory shapes what we can perceive, he challenged the notion of “pure” data and sparked enduring debates between realists and instrumentalists. His legacy reshaped the ontology of space‑time and the philosophy of science, leaving an indelible mark on twentieth‑century thought.


In [27]:
# Step 3: Come up with a few options for the title
prompt_step_3 = f"""
Based on the INTRODUCTION below, come up with 3 catchy blog post titles.
INTRODUCTION: {response_step_2}
Format your output as 3 bullet points.
""" 
response_step_3 = generate_response(
    model=model,
    tokenizer=tokenizer,
    user_prompt=prompt_step_3,
)

print(response_step_3)

- **“From Absolutes to Relativity: How Einstein’s Equations Turned Reality on Its Head”**  
- **“Realism vs. Instrumentalism: The Great Philosophic Debate Sparked by Relativity”**  
- **“Space‑Time’s New Ontology: What Einstein Taught Us About Observation and Determinism”**


Main motivation behind the prompt chaining technique: a few smaller prompts are better than one that is a giant one. If your application require solving complex tasks with multiple steps, divide them into smaller subtasks by introducing smaller prompts and chain LLM's outputs together with the following prompts.

Remember: When working with LLMs, simpler instructions are better than complex ones.

**Best Practice #3: Break complex prompts into smaller ones.**